In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [4]:
!pip install -q pycocotools transformers albumentations opencv-python tqdm
!pip install -q pytorch-lightning lightning torchmetrics faster-coco-eval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 51.5 MB/s eta 0:00:00


In [5]:
!git clone https://github.com/roboflow/rf-detr.git
%cd rf-detr
!pip install -e .

Cloning into 'rf-detr'...
remote: Enumerating objects: 17840, done.
remote: Counting objects: 100% (1162/1162), done.
remote: Compressing objects: 100% (665/665), done.
remote: Total 17840 (delta 893), reused 497 (delta 497), pack-reused 16678 (from 4)
Receiving objects: 100% (17840/17840), 28.79 MiB | 20.87 MiB/s, done.
Resolving deltas: 100% (10333/10333), done.
/content/rf-detr
Obtaining file:///content/rf-detr
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 20.4 MB/s eta 0:00:00
  Building editable for rfdetr (pyproject.toml) ... done
  Created wheel for rfdetr: filename=rfdetr-1.8.1-0.editable-py3-none-any.wh

In [6]:
#import rf -detr
import sys
sys.path.append("/content/rf-detr/src")

import rfdetr

print("RF-DETR imported successfully!")

RF-DETR imported successfully!


/content/rf-detr/src/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


In [7]:
#import training modules
from rfdetr.training import (
    RFDETRDataModule,
    RFDETRModelModule,
    build_trainer,
    seed_everything
)

from rfdetr.config import (
    ModelConfig,
    TrainConfig
)

seed_everything(42)

print("Training modules imported successfully!")

INFO:lightning_fabric.utilities.seed:Seed set to 42


Training modules imported successfully!


In [8]:
#copy dataset to colab
import shutil
import os

SRC = "/content/drive/MyDrive/data/version_1"
DST = "/content/rf_dataset"

if os.path.exists(DST):
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)

print("Dataset copied.")

Dataset copied.


In [9]:
#flatten dataset
for split in ["train", "valid", "test"]:
    images_dir = f"{DST}/{split}/images"

    if os.path.exists(images_dir):
        for file in os.listdir(images_dir):
            shutil.move(
                os.path.join(images_dir, file),
                os.path.join(f"{DST}/{split}", file)
            )

        os.rmdir(images_dir)

print("Dataset flattened.")

Dataset flattened.


In [10]:
#verify dataset
!find /content/rf_dataset/valid -maxdepth 1

/content/rf_dataset/valid
/content/rf_dataset/valid/phone_022_jpeg.rf.81a59de42f5f6d560fdbd8a86d07a418.jpg
/content/rf_dataset/valid/phone_040_jpeg.rf.684bcb28a5b5d2356d03bffc77ce2fbd.jpg
/content/rf_dataset/valid/phone_010_jpeg.rf.1c71eb9ecd0536258a84087cd0dee1ef.jpg
/content/rf_dataset/valid/phone_069_jpeg.rf.0238682e5ee1fff1e4380f407c7da735.jpg
/content/rf_dataset/valid/phone_079_jpeg.rf.9351b0aa32c05a1f6f47078437e6bd87.jpg
/content/rf_dataset/valid/phone_076_jpeg.rf.001a06dfd64c273e6646b6fa3c6c5ded.jpg
/content/rf_dataset/valid/phone_047_jpeg.rf.5afce60449aa89494045b6216b5bd773.jpg
/content/rf_dataset/valid/phone_077_jpeg.rf.9edea1ac6cc71a9459ca293c9bab3061.jpg
/content/rf_dataset/valid/phone_005_jpeg.rf.a19751d4941ba054ca25de8b89833796.jpg
/content/rf_dataset/valid/phone_066_jpeg.rf.78f0cacea3a61aa2428e3261a1775260.jpg
/content/rf_dataset/valid/phone_049_jpeg.rf.85dd4e18c04ff82ff81fdd78a2242b12.jpg
/content/rf_dataset/valid/phone_063_jpeg.rf.3cbb708348f4592e8660d5b1ae85a426.jpg
/c

In [11]:
#create configs
model_config = ModelConfig(
    encoder="dinov2_windowed_small",
    out_feature_indexes=[1, 2, 3],
    dec_layers=6,
    projector_scale=["P3", "P4", "P5"],
    hidden_dim=256,
    patch_size=14,
    num_windows=8,
    sa_nheads=8,
    ca_nheads=8,
    dec_n_points=4,
    resolution=256,
    positional_encoding_size=128,
    num_classes=1,
    segmentation_head=False,
    gradient_checkpointing=True,
    device="cuda"
)

train_config = TrainConfig(
    dataset_file="roboflow",
    dataset_dir="/content/rf_dataset",
    class_names=["mobile_phone"],
    batch_size=1,
    epochs=2,
    lr=1e-4,
    weight_decay=1e-4,
    output_dir="/content/output_debug",
    checkpoint_interval=1,
    eval_interval=1,
    num_workers=0,
    segmentation_head=False,
    accelerator="gpu",
    devices=1,
    seed=42,
    tensorboard=True,
    progress_bar="tqdm"
)

print("Configs created.")

Configs created.


In [12]:
#create datamodule
data_module = RFDETRDataModule(
    model_config=model_config,
    train_config=train_config
)

print("DataModule created.")

DataModule created.


In [13]:
#create model
model_module = RFDETRModelModule(
    model_config=model_config,
    train_config=train_config
)

print("ModelModule created.")

[2026-06-22 12:39:42] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


ModelModule created.


In [15]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache() #clear gpu memory

In [16]:
#build trainer
trainer = build_trainer(
    train_config=train_config,
    model_config=model_config,
    accelerator="gpu"
)

print("Trainer created.")

INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Trainer created.


In [17]:
#training
trainer.fit(
    model=model_module,
    datamodule=data_module
)

[2026-06-22 12:40:45] [INFO] rf-detr - Building Roboflow train dataset with square resize at resolution 256
[2026-06-22 12:40:45] [INFO] rf-detr - Using multi-scale training with square resize and scales: [784]
[2026-06-22 12:40:45] [INFO] rf-detr - Built 1 Albumentations transforms from config


[2026-06-22 12:40:45] [WARNING] rf-detr - Keypoint pipeline: 'HorizontalFlip' performs a horizontal flip but flip-pair swapping (swapping left/right joint labels after a horizontal flip) is not yet implemented. The transform has been disabled to prevent incorrect keypoint annotations. Remove 'HorizontalFlip' from your augmentation config or wait for flip-pair support in a future release.


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
[2026-06-22 12:40:45] [INFO] rf-detr - Building Roboflow val dataset with square resize at resolution 256
[2026-06-22 12:40:45] [INFO] rf-detr - Using multi-scale training with square resize and scales: [784]
[2026-06-22 12:40:45] [INFO] rf-detr - Built 1 Albumentations transforms from config
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/output_debug exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model       │ LWDETR       │ 49.6 M │ train │     0 │
│ 1 │ criterion   │ SetCriterion │      0 │ train │     0 │
│ 2 │ postprocess │ PostProcess  │      0 │ train │     0 │
└───┴─────────────┴──────────────┴────────┴───────┴───────┘

...

[2026-06-22 12:41:24] [INFO] rf-detr - Best regular checkpoint saved to /content/output_debug/checkpoint_best_regular.pth (epoch 0, monitor=val/mAP_50_95, value=0)


Validation: |          | 0/? [00:00<?, ?it/s]

[2026-06-22 12:42:07] [INFO] rf-detr - Best regular checkpoint saved to /content/output_debug/checkpoint_best_regular.pth (epoch 1, monitor=val/mAP_50_95, value=2.18634e-05)


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=2` reached.


[2026-06-22 12:42:29] [INFO] rf-detr - Best total checkpoint saved from regular (regular=0.0000, ema=0.0000)


In [18]:
!find /content/output_debug -type f

/content/output_debug/metrics.csv
/content/output_debug/hparams.yaml
/content/output_debug/events.out.tfevents.1782132045.e807a9246a69.1554.0
/content/output_debug/checkpoint_0.ckpt
/content/output_debug/checkpoint_1.ckpt
/content/output_debug/checkpoint_best_total.pth
/content/output_debug/checkpoint_best_regular.pth


In [19]:
#save outputs to drive
save_dir = "/content/drive/MyDrive/RF_DETR_Output"

os.makedirs(save_dir, exist_ok=True)

shutil.copytree(
    "/content/output_debug",
    f"{save_dir}/debug_training",
    dirs_exist_ok=True
)

print("Outputs saved to Drive.")

Outputs saved to Drive.
